# Acquisizione su matrice X-Y a meandro (laser + oscilloscopio + motori PI)

Per ogni punto di una matrice X-Y centrata sulla membrana: si muovono i due motori PI C-863, poi si fa **una scansione completa** del laser sincronizzata con lo scope (stessa sequenza di `acquisizione_laser_scope.ipynb`, quella che funziona), e si salva un CSV per punto.

- Motore **X** = `ASRL4` (COM4), motore **Y** = `ASRL5` (COM5). Limiti 0-150 mm.
- **Chiudi PI MicroMove** prima di partire: la porta seriale può essere usata da un solo programma.
- Matrice `N_X` x `N_Y` (8 x 8) centrata su `CENTRO_X_MM`, `CENTRO_Y_MM`, che copre il lato della membrana (`LATO_MEMBRANA_UM` = 100 µm) dal bordo al bordo, quindi con passo lato/(N-1). Percorso a meandro: si parte dalla riga in alto andando verso destra (X crescente), si scende di una riga e si torna verso sinistra, e così via. Se "destra" o "giù" sono invertiti sul tuo setup, cambia `DESTRA_E_X_CRESCENTE` e `GIU_E_Y_DECRESCENTE` in cella 1.
- Per punti non equidistanti letti da file usa `acquisizione_griglia_xy.ipynb`.

Ordine: celle 1-4 una volta; la cella 5 mostra l'anteprima dei punti (**non muove niente**); la cella 6 li scansiona tutti e chiede conferma prima di muovere i motori.

Attenzione allo spazio su disco: ogni CSV di un'acquisizione (5 milioni di campioni) pesa circa 240 MB.

In [1]:
# Cella 1 - configurazione e funzioni (scope, laser, griglia)
import os
import sys
import time
import datetime

import numpy as np
import pyvisa

OSC_VISA = "USB::0x0699::0x0408::C028813::INSTR"
OSC_TIMEOUT_MS = 30000

CHANNELS = ["CH1", "CH2"]

LASER_WAVELENGTH_START = 1520.0  # nm, come trigger_scan.m
LASER_WAVELENGTH_STOP = 1570.0   # nm, come trigger_scan.m

POLL_INTERVAL_S = 0.1
CHANNEL_READ_PAUSE_S = 0.1

# ---------------- matrice X-Y (in mm, limiti motori 0-150) ----------------
CENTRO_X_MM = 92.298
CENTRO_Y_MM = 146.543
LATO_MEMBRANA_UM = 100.0   # i punti vanno da -lato/2 a +lato/2 (bordi inclusi)
N_X = 8
N_Y = 8
DESTRA_E_X_CRESCENTE = True   # "destra" = X crescente
GIU_E_Y_DECRESCENTE = True    # "scendere" = Y decrescente; si parte dalla riga in alto

ASSESTAMENTO_S = 1.0           # pausa dopo il movimento, prima di acquisire
ATTESA_RITORNO_LASER_S = 5.0  # tra due scan, perché il laser torni a 1520 nm (CH1 basso)
RESETTA_CONTATORE_SCAN = True  # prima di ogni start: OUTP:SCAN:STOP + SOUR:WAVE:DESSCANS 1 (il laser potrebbe smettere di partire dopo N scan)
# --------------------------------------------------------------------------


def crea_cartella_run():
    oggi = datetime.date.today().strftime("%Y-%m-%d")
    base_folder = os.path.join(os.getcwd(), oggi)
    os.makedirs(base_folder, exist_ok=True)
    run_esistenti = [d for d in os.listdir(base_folder) if d.startswith("run_")]
    run_folder = os.path.join(base_folder, f"run_{len(run_esistenti) + 1:03d}")
    os.makedirs(run_folder)
    return run_folder


def configura_scope(osc):
    osc.write("HORizontal:MODE MANUAL")
    rec_len = int(osc.query("HORizontal:RECORDLENGTH?"))
    osc.write("DATA:START 1")
    osc.write(f"DATA:STOP {rec_len}")
    osc.write("DATA:WIDTH 1")
    osc.write("DATA:ENCdg ASCii")


def arm_acquisition(osc, timeout_s=120):
    osc.write("ACQ:STOPAFTER SEQUENCE")
    osc.write("ACQ:STATE RUN")
    # Lo scan parte solo quando lo scope e' READY (finito il pre-trigger).
    print("Attendo che lo scope finisca il pre-trigger (-> READY)...")
    t0 = time.time()
    ultimo = None
    while True:
        stato = osc.query("TRIGger:STATE?").strip().upper()
        if stato != ultimo:
            print(f"  stato trigger: {stato}")
            ultimo = stato
        if stato == "READY":
            break
        if time.time() - t0 > timeout_s:
            raise TimeoutError(
                f"Lo scope non e' arrivato a READY in {timeout_s} s "
                f"(stato trigger: {stato}, ACQ:STATE: {osc.query('ACQ:STATE?').strip()})"
            )
        time.sleep(POLL_INTERVAL_S)
    print("Scope pronto ad accettare il trigger.")


def wait_for_acquisition(osc):
    print("Attendo che l'oscilloscopio completi l'acquisizione triggerata...")
    while True:
        if int(osc.query("ACQ:STATE?")) == 0:
            print("Acquisizione completata!")
            break
        time.sleep(POLL_INTERVAL_S)


def acquire_channel(osc, source="CH1"):
    osc.write(f"DATA:SOURCE {source}")
    ymult = float(osc.query("WFMPRE:YMULT?"))
    yoff = float(osc.query("WFMPRE:YOFF?"))
    yzero = float(osc.query("WFMPRE:YZERO?"))
    print(f"{source}: YMULT={ymult}, YOFF={yoff}, YZERO={yzero}")
    raw = osc.query("CURVE?")
    ydata = np.array(raw.split(","), dtype=float)
    return (ydata - yoff) * ymult + yzero

In [2]:
# Cella 2 - connessione all'oscilloscopio (una volta per sessione)
rm = pyvisa.ResourceManager()
osc = rm.open_resource(OSC_VISA)
osc.timeout = OSC_TIMEOUT_MS
configura_scope(osc)
print("Oscilloscopio connesso e configurato.")

Oscilloscopio connesso e configurato.


In [3]:
# Cella 3 - laser: carica la DLL e apre il dispositivo (una volta per sessione)
USBADDR = 1
DEVICE_ID = 0x100A  # hex2dec('100A')

dll_path = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "scripts", "laser", "UsbDllWrap.dll"))
if not os.path.isfile(dll_path):
    raise RuntimeError(f"UsbDllWrap.dll non trovata in {dll_path} (il notebook va aperto da Chiara_Bru/runs_oscilloscope)")
print(f"DLL trovata: {dll_path}")
sys.path.append(os.path.dirname(dll_path))

import clr
clr.AddReference("UsbDllWrap")
import Newport
from System.Text import StringBuilder

NP_USB = Newport.USBComm.USB()

esito = NP_USB.OpenDevices(DEVICE_ID)
print(f"OpenDevices({hex(DEVICE_ID)}) -> {esito}")
querydata = StringBuilder(64)


def laser_query(comando):
    esito = NP_USB.Query(USBADDR, comando, querydata)
    risposta = querydata.ToString()
    print(f"Query {comando!r} -> esito={esito}, risposta={risposta!r}")
    return risposta


LASER_COMANDI_CON_QUERY = True  # True: come i driver di riferimento (Query + lettura dell'OK)


def laser_write(comando):
    if LASER_COMANDI_CON_QUERY:
        # Con Write la risposta "OK" del laser resta non letta e si accumula,
        # sfasando le query successive e saturando la coda del laser.
        esito = NP_USB.Query(USBADDR, comando, querydata)
        print(f"Write {comando!r} -> esito={esito}, risposta={querydata.ToString()!r}")
    else:
        esito = NP_USB.Write(USBADDR, comando)
        print(f"Write {comando!r} -> esito={esito}")


laser_query("*IDN?")
laser_query("OUTP:STAT?")

DLL trovata: c:\Users\Labo\Desktop\Tesi\scripts\laser\UsbDllWrap.dll
OpenDevices(0x100a) -> True
Query '*IDN?' -> esito=0, risposta='New_Focus 6700 v2.4 03/19/14 SN23163'
Query 'OUTP:STAT?' -> esito=0, risposta='1'


'1'

In [4]:
# Cella 4 - motori PI C-863: X = ASRL4, Y = ASRL5 (una volta per sessione)
MOTORE_X = "ASRL4::INSTR"
MOTORE_Y = "ASRL5::INSTR"
PI_BAUD = 19200
PI_AXIS = "1"
PI_TIMEOUT_MS = 2000
PI_TIMEOUT_MOVIMENTO_S = 60
PI_TOLLERANZA_MM = 0.01  # valore usato nella prova sul laboratorio (i motori arrivano entro circa 10 um)


def pi_apri(porta):
    inst = rm.open_resource(porta)
    inst.baud_rate = PI_BAUD
    inst.timeout = PI_TIMEOUT_MS
    inst.read_termination = "\n"
    inst.write_termination = "\n"
    return inst


def pi_query(inst, comando):
    return inst.query(comando).strip()


def pi_valore(inst, comando):
    """Estrae il numero da risposte tipo '1=+0012.3456'."""
    return float(pi_query(inst, comando).split("=")[-1])


def pi_limiti(inst):
    return pi_valore(inst, f"TMN? {PI_AXIS}"), pi_valore(inst, f"TMX? {PI_AXIS}")


def pi_muovi(inst, nome, target):
    """Movimento assoluto in mm, con controllo dei limiti prima di muovere."""
    tmin, tmax = pi_limiti(inst)
    if not (tmin <= target <= tmax):
        raise ValueError(f"Motore {nome}: target {target} mm fuori dai limiti {tmin}..{tmax} mm")
    inst.write(f"MOV {PI_AXIS} {target:.4f}")
    time.sleep(0.1)
    t0 = time.time()
    while True:
        in_target = pi_query(inst, f"ONT? {PI_AXIS}").endswith("1")
        pos = pi_valore(inst, f"POS? {PI_AXIS}")
        if in_target and abs(pos - target) <= PI_TOLLERANZA_MM:
            break
        if time.time() - t0 > PI_TIMEOUT_MOVIMENTO_S:
            raise TimeoutError(f"Motore {nome}: non arrivato a {target} mm (posizione {pos} mm)")
        time.sleep(0.1)
    errore = int(pi_query(inst, "ERR?"))
    if errore != 0:
        raise RuntimeError(f"Motore {nome}: ERR? = {errore} dopo il movimento a {target} mm")
    print(f"Motore {nome}: a {pos:.4f} mm")


motore_x = pi_apri(MOTORE_X)
motore_y = pi_apri(MOTORE_Y)

for nome, inst in (("X", motore_x), ("Y", motore_y)):
    errore_residuo = pi_query(inst, "ERR?")  # legge e azzera un eventuale errore vecchio
    tmin, tmax = pi_limiti(inst)
    print(f"Motore {nome}: {pi_query(inst, '*IDN?')}")
    print(f"   posizione {pi_valore(inst, f'POS? {PI_AXIS}')} mm | limiti {tmin}..{tmax} mm | "
          f"servo {pi_query(inst, f'SVO? {PI_AXIS}')} | referenziato {pi_query(inst, f'FRF? {PI_AXIS}')} | "
          f"errore residuo letto: {errore_residuo}")

Motore X: (c)2013 Physik Instrumente(PI) Karlsruhe, C-863.11,0,1.3.0.7
   posizione 92.2765 mm | limiti 0.0..150.0 mm | servo 1=1 | referenziato 1=1 | errore residuo letto: 0
Motore Y: (c)2013 Physik Instrumente(PI) Karlsruhe, C-863.11,0,1.3.0.7
   posizione 146.593 mm | limiti 0.0..150.0 mm | servo 1=1 | referenziato 1=1 | errore residuo letto: 0


In [5]:
# Cella 5 - costruisce la matrice a meandro e mostra l'ANTEPRIMA (NON muove niente)
off_x = np.linspace(-LATO_MEMBRANA_UM / 2, LATO_MEMBRANA_UM / 2, int(N_X)) / 1000.0  # mm
off_y = np.linspace(-LATO_MEMBRANA_UM / 2, LATO_MEMBRANA_UM / 2, int(N_Y)) / 1000.0  # mm
x_crescenti = [CENTRO_X_MM + d for d in off_x]
y_crescenti = [CENTRO_Y_MM + d for d in off_y]

righe_y = y_crescenti[::-1] if GIU_E_Y_DECRESCENTE else y_crescenti   # dalla riga in alto
verso_destra = x_crescenti if DESTRA_E_X_CRESCENTE else x_crescenti[::-1]

PUNTI = []
for j, y in enumerate(righe_y):
    riga = verso_destra if j % 2 == 0 else verso_destra[::-1]   # meandro
    PUNTI += [(float(x), float(y)) for x in riga]
print(f"Matrice {int(N_X)} x {int(N_Y)} a meandro, passo X = {LATO_MEMBRANA_UM / (N_X - 1):.2f} um, "
      f"passo Y = {LATO_MEMBRANA_UM / (N_Y - 1):.2f} um")

tutti_x = np.array([p[0] for p in PUNTI])
tutti_y = np.array([p[1] for p in PUNTI])
for nome, inst, valori in (("X", motore_x, tutti_x), ("Y", motore_y, tutti_y)):
    tmin, tmax = pi_limiti(inst)
    if valori.min() < tmin or valori.max() > tmax:
        raise ValueError(f"Punti {nome} fuori dai limiti {tmin}..{tmax} mm")

print(f"{len(PUNTI)} punti, limiti rispettati.")
for i, (x, y) in enumerate(PUNTI):
    print(f"  {i + 1:3d}: X={x:.4f}  Y={y:.4f}")
print(f"Spazio su disco stimato: circa {len(PUNTI) * 0.24:.1f} GB")

Matrice 8 x 8 a meandro, passo X = 14.29 um, passo Y = 14.29 um
64 punti, limiti rispettati.
    1: X=92.2480  Y=146.5930
    2: X=92.2623  Y=146.5930
    3: X=92.2766  Y=146.5930
    4: X=92.2909  Y=146.5930
    5: X=92.3051  Y=146.5930
    6: X=92.3194  Y=146.5930
    7: X=92.3337  Y=146.5930
    8: X=92.3480  Y=146.5930
    9: X=92.3480  Y=146.5787
   10: X=92.3337  Y=146.5787
   11: X=92.3194  Y=146.5787
   12: X=92.3051  Y=146.5787
   13: X=92.2909  Y=146.5787
   14: X=92.2766  Y=146.5787
   15: X=92.2623  Y=146.5787
   16: X=92.2480  Y=146.5787
   17: X=92.2480  Y=146.5644
   18: X=92.2623  Y=146.5644
   19: X=92.2766  Y=146.5644
   20: X=92.2909  Y=146.5644
   21: X=92.3051  Y=146.5644
   22: X=92.3194  Y=146.5644
   23: X=92.3337  Y=146.5644
   24: X=92.3480  Y=146.5644
   25: X=92.3480  Y=146.5501
   26: X=92.3337  Y=146.5501
   27: X=92.3194  Y=146.5501
   28: X=92.3051  Y=146.5501
   29: X=92.2909  Y=146.5501
   30: X=92.2766  Y=146.5501
   31: X=92.2623  Y=146.5501
   32: X

In [6]:
# Cella 6 - ESEGUE la griglia: per ogni punto muove X e Y, poi una scansione completa
if input(f"Muovere i motori ed eseguire {len(PUNTI)} scansioni? scrivi 'si' per confermare: ").strip().lower() != "si":
    raise RuntimeError("Annullato: nessun movimento eseguito.")

run_folder = crea_cartella_run()
print(f"I dati saranno salvati in: {run_folder}")
registro_path = os.path.join(run_folder, "griglia.csv")
with open(registro_path, "w") as f:
    f.write("indice,x_mm,y_mm,file\n")

t0p = time.time()


def tp(msg):
    print(f"   [{time.time() - t0p:6.1f} s] {msg}")


try:
    for idx, (x, y) in enumerate(PUNTI):
        print(f"\n=== Punto {idx + 1}/{len(PUNTI)}: X={x:.3f} mm, Y={y:.3f} mm ===")
        t0p = time.time()
        pi_muovi(motore_x, "X", x)
        pi_muovi(motore_y, "Y", y)
        tp("motori in posizione")
        time.sleep(ASSESTAMENTO_S)
        tp("assestamento finito")

        # stessa sequenza dell'altro notebook (acquisizione_laser_scope.ipynb)
        laser_write(f"SOUR:WAVE:START {LASER_WAVELENGTH_START}")
        laser_write(f"SOUR:WAVE:STOP {LASER_WAVELENGTH_STOP}")
        if RESETTA_CONTATORE_SCAN:
            laser_write("OUTP:SCAN:STOP")
            laser_write("SOUR:WAVE:DESSCANS 1")

        arm_acquisition(osc)
        tp("scope READY")
        laser_write("OUTP:SCAN:START")
        tp("start inviato al laser")

        wait_for_acquisition(osc)
        tp("acquisizione completata")
        time.sleep(1.0)

        data = {}
        for ch in CHANNELS:
            print(f"Leggo {ch}...")
            data[ch] = acquire_channel(osc, ch)
            time.sleep(CHANNEL_READ_PAUSE_S)
        tp("canali letti")

        nome_file = f"punto_{idx:03d}_x{x:.3f}_y{y:.3f}.csv"
        min_len = min(len(v) for v in data.values())
        all_cols = np.column_stack([data[ch][:min_len] for ch in CHANNELS])
        np.savetxt(os.path.join(run_folder, nome_file), all_cols, delimiter=",", header=",".join(CHANNELS), comments="")
        with open(registro_path, "a") as f:
            f.write(f"{idx},{x:.4f},{y:.4f},{nome_file}\n")
        print(f"Salvato: {nome_file}")
        tp("file salvato")

        if idx < len(PUNTI) - 1:
            print(f"Attendo {ATTESA_RITORNO_LASER_S:.0f} s che il laser torni a {LASER_WAVELENGTH_START} nm...")
            time.sleep(ATTESA_RITORNO_LASER_S)
            tp("fine attesa laser")
except BaseException:
    print("\nInterrotto: provo a fermare lo scan del laser.")
    try:
        laser_write("OUTP:SCAN:STOP")
    except Exception as e:
        print(f"Impossibile fermare lo scan: {e}")
    raise

print(f"\nGriglia completata: {len(PUNTI)} punti in {run_folder}")

I dati saranno salvati in: c:\Users\Labo\Desktop\Tesi\Chiara_Bru\runs_oscilloscope\2026-10-05\run_021

=== Punto 1/64: X=92.248 mm, Y=146.593 mm ===
Motore X: a 92.2528 mm
Motore Y: a 146.5930 mm
   [   0.4 s] motori in posizione
   [   1.4 s] assestamento finito
Write 'SOUR:WAVE:START 1520.0' -> esito=0, risposta='OK'
Write 'SOUR:WAVE:STOP 1570.0' -> esito=0, risposta='OK'
Write 'OUTP:SCAN:STOP' -> esito=0, risposta='OK'
Write 'SOUR:WAVE:DESSCANS 1' -> esito=0, risposta='OK'
Attendo che lo scope finisca il pre-trigger (-> READY)...
  stato trigger: READY
Scope pronto ad accettare il trigger.
   [   1.4 s] scope READY
Write 'OUTP:SCAN:START' -> esito=0, risposta='OK'
   [   1.4 s] start inviato al laser
Attendo che l'oscilloscopio completi l'acquisizione triggerata...
Acquisizione completata!
   [  12.1 s] acquisizione completata
Leggo CH1...
CH1: YMULT=0.2, YOFF=-75.0, YZERO=0.0
Leggo CH2...
CH2: YMULT=0.0004, YOFF=-15.5, YZERO=0.0
   [  13.7 s] canali letti
Salvato: punto_000_x92.248